# Stateful Applications with CrewAI

This notebook demonstrates building a **stateful culinary assistant** using CrewAI that finds recipes based on ingredients and dietary restrictions, then provides step-by-step cooking instructions.

## What You'll Learn

- How to create a CrewAI **Agent** with a specific role, goal, and backstory
- How to define **Tasks** that chain together using the `context` parameter (stateful handoff)
- How to assemble a **Crew** and execute it with automatic planning

## Key Concept: Statefulness via Task Context

CrewAI supports statefulness through **task context** — a downstream task can reference the output of an upstream task, allowing the agent to carry forward prior results without manual wiring.

## Step 1: Import Dependencies & Configure the LLM

We import CrewAI's core building blocks — `Agent`, `Task`, `Crew` — and the `LLM` wrapper that lets us specify which language model powers our agent.

In [ ]:
import os
from crewai import Agent, Task, Crew, LLM

from dotenv import load_dotenv
load_dotenv()

In [ ]:
# Use CrewAI's native OpenAI provider pointed at Groq's API.
# "provider" must be explicit so CrewAI doesn't strip the "openai/" prefix
# from the model name — Groq requires the full name "openai/gpt-oss-20b".
llm = LLM(
    model="openai/gpt-oss-20b",
    provider="openai",
    base_url="https://api.groq.com/openai/v1",
    api_key=os.environ["GROQ_API_KEY"],
    temperature=0,
)

In [ ]:
# llm = ChatGroq(temperature=0,
#              model_name="openai/gpt-oss-20b")
# llm.invoke("Hello, how are you?").content

## Step 2: Define User Inputs

These variables act as the "user request" — the ingredient they want to cook with and any dietary restrictions. Try changing these values to see how the agent adapts!

In [ ]:
main_ingredient = "tomato"          # Primary ingredient the recipe must include
dietary_restrictions = "shrimps"    # Dietary constraint (e.g., allergens or preferences)

## Step 3: Create the Agent

A CrewAI **Agent** is defined by three key attributes:
- **`role`** — The persona the agent assumes (e.g., "Culinary Assistant")
- **`goal`** — What the agent is trying to achieve
- **`backstory`** — Context that shapes how the agent reasons and responds

Setting `verbose=True` lets us observe the agent's chain-of-thought during execution.

In [ ]:
culinary_assistant = Agent(
    llm=llm,
    role="Culinary Assistant",
    backstory=(
        "An experienced culinary assistant skilled in finding and tailoring "
        "recipes based on ingredients and dietary needs, and providing clear, "
        "step-by-step cooking instructions"
    ),
    goal="Find recipes, filter them to meet dietary preferences, and guide user through recipe steps.",
    verbose=True,  # Prints the agent's reasoning steps
)

## Step 4: Define Tasks (with Stateful Context)

Each **Task** tells an agent *what* to do (`description`) and *what output to produce* (`expected_output`).

The `context` parameter on the second task is what makes this **stateful** — it passes the output of `find_and_filter_recipes` into `guide_recipe_steps`, so the agent builds on its previous work rather than starting from scratch.

In [ ]:
# Task 1: Find and filter recipes based on user inputs
find_and_filter_recipes = Task(
    description=(
        f"Find recipes that use the ingredient: {main_ingredient} and "
        f"filter them to meet dietary restrictions: {dietary_restrictions}."
    ),
    expected_output=f"One recipe using {main_ingredient} and matching {dietary_restrictions} restrictions.",
    agent=culinary_assistant,
)

# Task 2: Generate step-by-step cooking instructions
# 'context' links this task to the output of Task 1 — this is the stateful handoff
guide_recipe_steps = Task(
    description="Provide step-by-step instructions for the selected recipe.",
    expected_output="Step-by-step cooking instructions for the chosen recipe.",
    agent=culinary_assistant,
    context=[find_and_filter_recipes],  # <-- Receives Task 1's output automatically
)

## Step 5: Assemble the Crew & Execute

The **Crew** brings agents and tasks together. Setting `planning=True` enables CrewAI's built-in planner, which generates an execution plan before running the tasks.

When `kickoff()` is called, the crew:
1. Plans the execution order
2. Runs `find_and_filter_recipes` to find a suitable recipe
3. Passes that result into `guide_recipe_steps` via the `context` link
4. Returns the final cooking instructions

In [ ]:
crew = Crew(
    agents=[culinary_assistant],
    tasks=[find_and_filter_recipes, guide_recipe_steps],
    planning=True,
    planning_llm=llm,  # Use our Groq LLM for planning too (defaults to OpenAI otherwise)
)

result = crew.kickoff()
print(result)

In [ ]:
print(result)